##Installations

Run this block to install the required libraries for processing, vectorization, and generation.

In [29]:
# had to download older langchain because ragas still uses the olde langchain-community
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.43 langchain-huggingface==0.0.3 langchain-ollama==0.1.3 pypdf chromadb sentence-transformers deepeval pandas ragas datasets streamlit

## Multi-Format Document Processing & Database Precomputation
This block loads the raw documents into memory exactly once, and then iterates through different chunk sizes (500, 1000, 2000) to build three separate ChromaDB vector stores on the local disk.

In [3]:
import os
import glob
import json
import warnings
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader, CSVLoader # langchain is restructuring (splitting this package into smaller ones) so this might has a warning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/RAG_Project"
KB_PATH = f"{DRIVE_PROJECT_PATH}/knowledge_base/*"

# 1. Load Raw Documents Once
def load_raw_documents(kb_folder=KB_PATH):
    print("Loading raw documents into memory...")
    all_files = glob.glob(kb_folder, recursive=True)
    raw_docs = []

    for file_path in all_files:
        if not os.path.isfile(file_path):
            continue

        file_name = os.path.basename(file_path)
        try:
            if file_path.endswith('.pdf'):
                raw_docs.extend(PyPDFLoader(file_path).load())
            elif file_path.endswith('.csv'):
                raw_docs.extend(CSVLoader(file_path).load())
            elif file_path.endswith('.json'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    text_content = json.dumps(json.load(f), indent=2)
                    raw_docs.append(Document(page_content=text_content, metadata={"source": file_name}))
        except Exception as e:
            print(f"Error loading {file_name}: {e}")

    print(f"Loaded {len(raw_docs)} raw document pages.")
    return raw_docs

raw_documents = load_raw_documents()

# 2. Setup Embedding Model
# warning might appear because the saved model weights include static 'position_ids', but the modern
# transformers library generates them dynamically at runtime, creating a safe mismatch.
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 3. Chunk and Build Databases for Multiple Sizes
chunk_sizes = [500, 1000, 2000]
vector_stores = {}

for size in chunk_sizes:
    print(f"\n--- Processing Chunk Size: {size} ---")
    overlap = int(size * 0.15)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)

    chunks = text_splitter.split_documents(raw_documents)
    print(f"Generated {len(chunks)} chunks.")

    persist_dir = f"{DRIVE_PROJECT_PATH}/chroma_db_{size}"
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_dir
    )
    vector_stores[size] = vector_store
    print(f"ChromaDB saved to {persist_dir}")

# 4. Set the 1000-chunk database as the baseline for downstream evaluation blocks
vector_store_baseline = vector_stores[1000]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading raw documents into memory...
Loaded 413 raw document pages.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Processing Chunk Size: 500 ---
Generated 2382 chunks.
ChromaDB saved to /content/drive/MyDrive/RAG_Project/chroma_db_500

--- Processing Chunk Size: 1000 ---
Generated 1281 chunks.
ChromaDB saved to /content/drive/MyDrive/RAG_Project/chroma_db_1000

--- Processing Chunk Size: 2000 ---
Generated 732 chunks.
ChromaDB saved to /content/drive/MyDrive/RAG_Project/chroma_db_2000


## The RAG System (Native Baseline)
Maps the query to the baseline retrieval input and uses the local LLM to output concise guidelines.

In [5]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Initialize the local LLM
llm = OllamaLLM(model="llama3")

def build_native_rag_chain(vector_store, top_k=3):
    retriever = vector_store.as_retriever(search_kwargs={"k": top_k})

    system_prompt = (
        "You are an expert incident response practitioner. "
        "Use the following retrieved context to generate 3 to 5 concise guidelines. "
        "CRITICAL RULE 1: You MUST cite the exact source document name (e.g., [filename.pdf]) for every step. Do not just use numbers like [1]. "
        "CRITICAL RULE 2: Do NOT recommend expensive enterprise tools like SIEMs or assume the user has a massive SOC team. Keep advice universally applicable for low-resource environments. "
        "Context:\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    return rag_chain

# Execute baseline RAG setup (Top-K = 3)
native_rag_chain = build_native_rag_chain(vector_store_baseline, top_k=3)

## Local Server Initialization
Boot up the local Ollama server in the background and pull the Llama 3 weights.

In [32]:
# 1. Install missing dependencies (zstd for extraction, pciutils for GPU)
!sudo apt install -y zstd pciutils > /dev/null

# 2. Install Ollama natively
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the server completely detached from the Colab cell
!nohup ollama serve > ollama_server.log 2>&1 </dev/null &

# 4. Give the server a few seconds to wake up
!sleep 5

# 5. Download the Llama 3 model
!ollama pull llama3



>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



## Automated Evaluation (DeepEval)
Runs the evaluation queries through the baseline pipeline and scores the outputs using local custom metrics.

In [7]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import OllamaModel
from deepeval import evaluate

print("Connecting to local Llama 3 server...")

local_judge_model = OllamaModel(
    model="llama3",
    base_url="http://localhost:11434",
    temperature=0
)

inclusivity_metric = GEval(
    name="Inclusivity and Resource Bias",
    criteria="Determine if the generated guidelines are universally applicable or if they possess a resource bias (e.g., assuming the user has access to expensive enterprise SIEM tools, massive SOC teams, etc.).",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=local_judge_model,
)

transparency_metric = GEval(
    name="Source Transparency",
    criteria="Determine whether the actual output explicitly cites the source documents provided in the retrieval context.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT],
    model=local_judge_model,
)

eval_queries = [
    "What are the immediate containment steps for a compromised cloud server?",
    "How do we safely capture volatile memory during an active breach?",
    "What is the communication protocol when a massive ransomware event occurs?"
]

deep_eval_test_cases = []
print("Running evaluation queries through Native RAG...")

for query in eval_queries:
    response = native_rag_chain.invoke({"input": query})

    test_case = LLMTestCase(
        input=query,
        actual_output=response["answer"],
        retrieval_context=[doc.page_content for doc in response["context"]]
    )
    deep_eval_test_cases.append(test_case)

print(f"Generated {len(deep_eval_test_cases)} test cases. Starting DeepEval scoring...")

# Capturing output to dummy variable to keep notebook clean
_ = evaluate(deep_eval_test_cases, metrics=[inclusivity_metric, transparency_metric])

Connecting to local Llama 3 server...
Running evaluation queries through Native RAG...
Generated 3 test cases. Starting DeepEval scoring...


✨ You're running DeepEval's latest Inclusivity and Resource Bias [GEval] Metric! (using llama3 (Ollama), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Source Transparency [GEval] Metric! (using llama3 (Ollama), strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            What are the immediate containment steps for a compromised cloud server?               │
│  │     Actual Output:    Based on the provided context and considering low-resource environments, here are      │
│  │                       3-5 concise guidelines for immediate containment of a compromised cloud server:        │
│  │                                                                                                              │
│  │                       **1. Isolate the affected server immediately (source: [Cloud Initial Infection         │
│  │                       Vectors, 2025])**                                                                      │
│  │                       Disconnect the compromised server from all networks to prevent further lateral         │
│  │                       movement and data exfiltration.                                                        │
│  │                                                                                                              │
│  │                       **2. Gather critical information before conducting further actions (no specific        │
│  │                       source document cited)**                                                               │
│  │                       Collect relevant details about the compromise, such as:                                │
│  │                               * The type of cloud service affected                                           │
│  │                               * The approximate time of detection                                            │
│  │                               * Any unusual user behavior or login attempts                                  │
│  │                               * Any suspicious files or directories created                                  │
│  │                                                                                                              │
│  │                       **3. Conduct a thorough system check and identify potential entry points (source:      │
│  │                       [Cloud Compromises])**                                                                 │
│  │                       Review the server's configuration, logs, and system state to:                          │
│  │                               * Identify potential vulnerabilities exploited by attackers                    │
│  │                               * Detect any unusual or unauthorized changes                                   │
│  │                               * Validate the integrity of critical system files and directories              │
│  │                                                                                                              │
│  │                       **4. Implement basic security measures to prevent further exploitation (no specific    │
│  │                       source document cited)**                                                               │
│  │                       Apply temporary security controls to limit the attacker's movement, such as:           │
│  │                               * Disabling remote access protocols like RDP or SSH                            │
│  │                               * Restricting network t

⚠ WARNING: No hyperparameters logged.
» ]8;id=508687;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 47.02s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Extended Evaluation: RAGAS Metrics
Reformats the test cases and evaluates them for contextual precision and answer relevancy.

In [12]:
from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import answer_relevancy, faithfulness

print("Formatting data for RAGAS...")

# 1. Extract the data already generated during the DeepEval loop
ragas_data = {
    "question": [tc.input for tc in deep_eval_test_cases],
    "answer": [tc.actual_output for tc in deep_eval_test_cases],
    "contexts": [tc.retrieval_context for tc in deep_eval_test_cases],
}

dataset = Dataset.from_dict(ragas_data)

print("Running RAGAS metrics...")

# 2. Evaluate using the same local LLM and Embeddings defined in Block 4
ragas_results = ragas_evaluate(
    dataset=dataset,
    metrics=[answer_relevancy, faithfulness],
    llm=llm,
    embeddings=embedding_model,
)

# 3. Display Results
df_ragas = ragas_results.to_pandas()
display(df_ragas)

/tmp/ipykernel_19236/50634214.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_relevancy, faithfulness
/tmp/ipykernel_19236/50634214.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import answer_relevancy, faithfulness


Formatting data for RAGAS...
Running RAGAS metrics...


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,answer_relevancy,faithfulness
0,What are the immediate containment steps for a...,[Stolen \nCredentials\nEmail\nPhishing\nInside...,Based on the provided context and considering ...,0.981060,0.000000
1,How do we safely capture volatile memory durin...,[develop procedures based on those discussions...,"Based on the provided context, here are three ...",0.963088,0.625000
2,What is the communication protocol when a mass...,"[OUTLOOK\nIn 2026, ransomware will remain a cr...","Based on the provided context, here are some c...",0.911408,0.272727


## 7. Streamlit Application Script
Writes the interactive dashboard logic (with chunk size toggling) to a local `app.py` file.

In [23]:
%%writefile app.py
import streamlit as st
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# --- UI CONFIGURATION ---
st.set_page_config(page_title="RAG Dashboard", layout="wide")
st.title("Incident Response RAG Assistant")

# --- CACHED RESOURCES ---
@st.cache_resource
def load_models():
    llm = Ollama(model="llama3", base_url="http://localhost:11434", temperature=0)
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    return llm, embeddings

llm, embeddings = load_models()

@st.cache_resource
def get_vectorstore(chunk_size):
    # Point Streamlit to read the database from Google Drive
    drive_path = f"/content/drive/MyDrive/RAG_Project/chroma_db_{chunk_size}"
    return Chroma(persist_directory=drive_path, embedding_function=embeddings)

# --- SIDEBAR CONTROLS ---
st.sidebar.header("Pipeline Settings")
selected_chunk_size = st.sidebar.selectbox("Chunk Size", [500, 1000, 2000], index=1)
top_k = st.sidebar.slider("Top-K Retrieved Chunks", min_value=1, max_value=10, value=3)
vector_store = get_vectorstore(selected_chunk_size)

# --- MAIN CHAT INTERFACE ---
st.subheader("Chat with the Knowledge Base")
if "messages" not in st.session_state:
    st.session_state.messages = []

# 1. Render chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

        # If this historical message has context, render the expander
        if "context" in msg and msg["context"]:
            with st.expander("🔍 View Retrieved Context Chunks"):
                for i, doc in enumerate(msg["context"]):
                    st.markdown(f"**Chunk {i+1} | Source: `{doc.metadata.get('source', 'Unknown')}`**")
                    st.caption(doc.page_content)
                    st.divider()

if prompt := st.chat_input("Enter your incident response query..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner(f"Querying DB (Chunk: {selected_chunk_size}, Top-K: {top_k})..."):
            retriever = vector_store.as_retriever(search_kwargs={"k": top_k})
            system_prompt = (
                "You are an expert incident response practitioner. "
                "Use the following retrieved context to answer the query concisely. "
                "CRITICAL RULE: You MUST cite the exact source document name (e.g., [filename.pdf]). "
                "Context:\n{context}"
            )
            qa_prompt = ChatPromptTemplate.from_messages([
                ("system", system_prompt),
                ("human", "{input}")
            ])
            question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
            rag_chain = create_retrieval_chain(retriever, question_answer_chain)

            response = rag_chain.invoke({"input": prompt})
            answer = response["answer"]
            context_chunks = response.get("context", [])

            st.markdown(answer)

            # 2. Render the chunks retrieved
            if not context_chunks:
                st.warning("⚠️ No context chunks retrieved. Your database might be empty or Google Drive is disconnected.")
            else:
                with st.expander("🔍 View Retrieved Context Chunks"):
                    for i, doc in enumerate(context_chunks):
                        st.markdown(f"**Chunk {i+1} | Source: `{doc.metadata.get('source', 'Unknown')}`**")
                        st.caption(doc.page_content)
                        st.divider()

    # 3. Save context to history (prevent UI bug affecting session)
    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "context": context_chunks
    })

Overwriting app.py


## Booting the Streamlit UI
### Instruction:
- Run the code block below, which will give a website URL and an IP address (IP will also show in the URL itself)
- Click on the URL and paste the IP into the prompted box
- Wait for the page to fully load and you can ask the Chatbot any domain-related questions, and adjust top-k or chunk size (currently only 500, 1000 and 2000)

Note: Sometimes the top-k and chunk size settings have display errors, just refresh the page till it loads correctly

In [33]:
# 1. Download Cloudflare's lightweight tunneling tool
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 2. Start Streamlit in the background
!nohup streamlit run app.py \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > streamlit.log 2>&1 &

# 3. Open the secure Cloudflare tunnel
import time
print("Starting Cloudflare Tunnel...")
time.sleep(3) # Give Streamlit a second to boot
!./cloudflared tunnel --url http://localhost:8501

Starting Cloudflare Tunnel...
2026-05-25T11:12:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-25T11:12:52Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-25T11:12:57Z INF +--------------------------------------------------------------------------------------------+
2026-05-25T11:12:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-25T11:12:57Z INF |  https://cameron-ideas-c